# Old features vs. new features: 30-minute speed forecasts on I-5 northbound (District 12)

This notebook compares the feature set from `data_and_features.ipynb` (**old**) with a revised set (**new**), using the same
data, the same model, and the same time-based folds.

**What changes from old to new**

| | Old | New |
|---|---|---|
| Training targets | every row, including speeds PeMS filled in | only future speeds that were actually measured |
| Current conditions | speed, change in speed over 10 min | + occupancy and flow |
| Neighbors | upstream and downstream speed | + upstream and downstream occupancy |
| Ramps | nearest on-ramp and off-ramp flows, one column per tied ramp; ramp speeds | tied ramps summed into one column; ramp speeds dropped (never reported) |
| When and where | none | time of day, day of week, postmile |
| History | none | typical speed at this station 30 minutes ahead (from training data only) |

**How it's judged.** Every model is scored only on future speeds PeMS actually **measured** (`pct_observed == 100`).
Scoring on filled-in speeds would mean grading the model on PeMS's estimates. Two numbers are reported:

- **MAE**: mean absolute error in mph over all measured targets.
- **MAE when changing**: the same, but only where speed changed by at least 10 mph in those 30 minutes. That's where a forecast is useful, and where it's hard.

Two simple guesses are included for reference: **same as now** (speed won't change) and **typical for this time** (this station's
usual speed at that time of day, computed from training data).

**Model:** scikit-learn `HistGradientBoostingRegressor` with the same settings for every feature set.

**The July–December 2025 holdout is never touched.** Files dated on or after `HOLDOUT_START` are skipped when the data is
loaded, so nothing in this notebook (models, typical-speed profiles, charts) sees them. Everything is judged on the four
validation windows only. Save the holdout for the final model.

**Run time.** Each fit uses up to `TRAIN_SAMPLE` training rows. With the defaults, the whole notebook takes roughly 15–30 minutes on a laptop.
Set `RUN_STEPS = False` to compare only old vs. new, which is about 3 times faster.

In [ ]:
# ---- Settings -------------------------------------------------------------
from pathlib import Path
DATA_DIR = Path("./Data")                                   # contains "D12 2023", "D12 2024", "D12 2025"
META_FILE = DATA_DIR / "d12_text_meta_2021_05_18.txt"
HOUR_START, HOUR_END = 6, 19          # keep 6:00-19:55
HORIZON_MIN = 30
CHANGE_MPH = 10                       # "changing" = speed moves at least this much over the horizon
TRAIN_SAMPLE = 2_000_000              # max training rows per fit (None = use all; slower)
RUN_STEPS = True                      # also fit the in-between feature sets (shows what each change adds)

TRAIN_START = "2023-01-01"
VAL_WINDOWS = [("2023-07-01", "2024-01-01"), ("2024-01-01", "2024-07-01"),
               ("2024-07-01", "2025-01-01"), ("2025-01-01", "2025-07-01")]
HOLDOUT_START = "2025-07-01"          # files on or after this date are never read

MODEL_PARAMS = dict(max_iter=300, learning_rate=0.05, max_leaf_nodes=31, min_samples_leaf=50,
                    early_stopping=False, random_state=0)

# Freeway-to-freeway connectors that act as northbound I-5 entrances/exits (from data_and_features.ipynb)
FF_ROLES = {1208990: "exit", 1209013: "exit", 1209012: "entrance", 1210020: "exit",
            1205315: "exit", 1212015: "entrance", 1213419: "exit"}
# ---------------------------------------------------------------------------

import time
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pandas.tseries.holiday import USFederalHolidayCalendar
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance

pd.set_option("display.width", 160)
HORIZON = pd.Timedelta(minutes=HORIZON_MIN)
KEY = ["station", "timestamp"]

BLUE, ORANGE, AQUA, YELLOW = "#2a78d6", "#eb6834", "#1baf7a", "#eda100"
INK, INK2, GRID = "#0b0b0b", "#52514e", "#e6e5e1"
plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "axes.edgecolor": GRID,
    "axes.labelcolor": INK2, "axes.titlecolor": INK, "axes.grid": True, "grid.color": GRID,
    "grid.linewidth": 0.8, "axes.axisbelow": True, "axes.spines.top": False, "axes.spines.right": False,
    "xtick.color": INK2, "ytick.color": INK2, "lines.linewidth": 2, "legend.frameon": False,
    "figure.dpi": 110, "axes.titlesize": 12,
})

## 1. Load the data
Weekdays only, with federal holidays removed (the old notebook kept holidays), from 6:00 to 19:55. Each file is read once, and mainline and ramp rows are split from it. Holdout files are skipped before reading.

In [ ]:
holidays = set(USFederalHolidayCalendar().holidays("2015-01-01", "2035-12-31").date)
files = sorted(DATA_DIR.rglob("d12_text_station_5min_*.parquet"))
COLS = ["timestamp", "station", "lane_type", "speed", "flow", "occupancy", "pct_observed"]

holdout_start = pd.Timestamp(HOLDOUT_START).date()
main_parts, conn_parts, used, skipped_holdout = [], [], 0, 0
t0 = time.time()
for i, path in enumerate(files, 1):
    day = datetime.strptime(path.stem[-10:], "%Y_%m_%d").date()
    if day >= holdout_start:
        skipped_holdout += 1
        continue                          # never read the holdout
    if day.weekday() >= 5 or day in holidays:
        continue
    x = pd.read_parquet(path, columns=COLS)
    x = x[x["timestamp"].dt.hour.between(HOUR_START, HOUR_END)]
    main_parts.append(x[x["lane_type"] == "ML"].drop(columns="lane_type"))
    conn_parts.append(x[x["lane_type"].isin(["OR", "FR"]) | x["station"].isin(FF_ROLES)].drop(columns="lane_type"))
    used += 1
    if i % 100 == 0:
        print(f"read {i}/{len(files)} files ({time.time() - t0:.0f}s)")

def finish(parts):
    df = pd.concat(parts, ignore_index=True)
    for c in ["speed", "flow", "occupancy"]:
        df[c] = df[c].astype("float32")
    df["real"] = df["pct_observed"] >= 100
    return df.drop(columns="pct_observed")

main, conn = finish(main_parts), finish(conn_parts)
del main_parts, conn_parts
assert main["timestamp"].max() < pd.Timestamp(HOLDOUT_START), "holdout data was loaded"
print(f"{used} weekdays used out of {len(files)} files ({skipped_holdout} holdout files skipped, never read) | "
      f"mainline rows {len(main):,} | ramp rows {len(conn):,}")
print("Share of mainline readings that are measured, by year:",
      main.groupby(main["timestamp"].dt.year)["real"].mean().round(2).to_dict())

## 2. Station order and nearest ramps
This is the same logic as the old notebook: the nearest on-ramp at or upstream of each mainline station, and the nearest off-ramp at or downstream, keeping every ramp tied at the same postmile.

In [ ]:
meta = pd.read_csv(META_FILE, sep="\t")
meta = meta[(meta["Fwy"] == 5) & meta["Dir"].isin(["N", "S"])].copy()
meta["Abs_PM"] = pd.to_numeric(meta["Abs_PM"], errors="coerce")
meta = meta.dropna(subset=["Abs_PM"])
meta["role"] = meta["Type"].map({"OR": "entrance", "FR": "exit"})
is_ff = meta["Type"].eq("FF")
meta.loc[is_ff, "role"] = meta.loc[is_ff, "ID"].map(FF_ROLES)
meta["travel_pm"] = np.where(meta["Dir"] == "N", meta["Abs_PM"], -meta["Abs_PM"])   # increases along travel

mls = meta[(meta["Type"] == "ML") & meta["ID"].isin(main["station"].unique())].sort_values(["Dir", "travel_pm"]).copy()
mls["upstream_id"] = mls.groupby("Dir")["ID"].shift(1)
mls["downstream_id"] = mls.groupby("Dir")["ID"].shift(-1)

rows = []
for _, s in mls.iterrows():
    road = meta[meta["Dir"] == s["Dir"]]
    on = road[(road["role"] == "entrance") & (road["travel_pm"] <= s["travel_pm"])]
    off = road[(road["role"] == "exit") & (road["travel_pm"] >= s["travel_pm"])]
    on = on[on["travel_pm"] == on["travel_pm"].max()]
    off = off[off["travel_pm"] == off["travel_pm"].min()]
    rows.append({"station": s["ID"], "on_ids": sorted(on["ID"]), "off_ids": sorted(off["ID"])})
ramps = pd.DataFrame(rows)
print(f"{len(mls)} mainline stations | stations with tied nearest on-ramps: {(ramps['on_ids'].str.len() > 1).sum()}, "
      f"tied off-ramps: {(ramps['off_ids'].str.len() > 1).sum()}")

## 3. Build all features once
The target is the speed 30 minutes later at the same station, along with whether that future speed was measured (`y_real`).

In [ ]:
d = main[KEY + ["speed", "occupancy", "flow", "real"]].copy()
fut = d[KEY + ["speed", "real"]].rename(columns={"speed": "y", "real": "y_real"})
fut["timestamp"] -= HORIZON
d = d.merge(fut, on=KEY, how="inner")
d["target_time"] = d["timestamp"] + HORIZON

# Change in speed over the last 10 minutes (old feature, kept)
prev = d[KEY + ["speed"]].rename(columns={"speed": "prev_speed"})
prev["timestamp"] += pd.Timedelta(minutes=10)
d = d.merge(prev, on=KEY, how="left")
d["speed_delta_2"] = d["speed"] - d["prev_speed"]
d = d.drop(columns="prev_speed")

# Ramp flows
cflow = conn[KEY + ["flow"]].rename(columns={"station": "rid"})
def ramp_lookup(ids):          # ids: DataFrame with columns station, rid -> flow aligned with d
    x = d[KEY].merge(ids, on="station", how="left").merge(cflow, on=["rid", "timestamp"], how="left")
    return x["flow"].to_numpy()

OLD_RAMP_COLS = []
for list_col, prefix in [("on_ids", "upstream_onramp"), ("off_ids", "downstream_offramp")]:
    ex = ramps[["station", list_col]].explode(list_col).dropna().rename(columns={list_col: "rid"})
    ex["n"] = ex.groupby("station").cumcount() + 1
    for n in range(1, ex["n"].max() + 1):                     # old: one column per tied ramp
        col = f"{prefix}_{n}_flow"
        d[col] = ramp_lookup(ex.loc[ex["n"] == n, ["station", "rid"]])
        OLD_RAMP_COLS.append(col)
d["onramp_flow_up"] = d[[c for c in OLD_RAMP_COLS if c.startswith("upstream_onramp")]].sum(axis=1, min_count=1)
d["offramp_flow_down"] = d[[c for c in OLD_RAMP_COLS if c.startswith("downstream_offramp")]].sum(axis=1, min_count=1)

# Neighboring mainline stations: speed (old) and occupancy (new)
nb = main[KEY + ["speed", "occupancy"]]
for col, name in [("upstream_id", "upstream"), ("downstream_id", "downstream")]:
    lk = mls[["ID", col]].rename(columns={"ID": "station", col: "nb"})
    x = d[KEY].merge(lk, on="station", how="left").merge(
        nb.rename(columns={"station": "nb", "speed": f"{name}_speed", "occupancy": f"{name}_occ"}),
        on=["nb", "timestamp"], how="left")
    d[f"{name}_speed"] = x[f"{name}_speed"].to_numpy()
    d[f"{name}_occ"] = x[f"{name}_occ"].to_numpy()

# When and where
d["tod"] = (d["timestamp"].dt.hour * 60 + d["timestamp"].dt.minute).astype("int16")
d["dow"] = d["timestamp"].dt.dayofweek.astype("int8")
d["pm"] = d["station"].map(mls.set_index("ID")["Abs_PM"]).astype("float32")
d["changing"] = (d["y"] - d["speed"]).abs() >= CHANGE_MPH

d = d.sort_values(KEY).reset_index(drop=True)
print(f"{len(d):,} rows with a 30-minute target | measured targets: {d['y_real'].mean():.0%}")
print(f"Memory: {d.memory_usage(deep=True).sum() / 1e9:.2f} GB")

### Feature sets
`typical_in_30` is computed separately for each fold, using only the training period, so the validation and test periods can't leak into it.

In [ ]:
OLD = ["speed", "speed_delta_2"] + OLD_RAMP_COLS + ["upstream_speed", "downstream_speed"]
OCC = ["occupancy", "flow", "upstream_occ", "downstream_occ"]
WHEN_WHERE = ["tod", "dow", "pm"]
NEW = (["speed", "speed_delta_2", "onramp_flow_up", "offramp_flow_down", "upstream_speed", "downstream_speed"]
       + OCC + WHEN_WHERE + ["typical_in_30"])

SETS = {"old": (OLD, False)}
if RUN_STEPS:
    SETS.update({
        "old, trained on measured targets": (OLD, True),
        "+ occupancy & flow": (OLD + OCC, True),
        "+ time, day, postmile": (OLD + OCC + WHEN_WHERE, True),
    })
SETS["new"] = (NEW, True)
for name, (cols, real_only) in SETS.items():
    print(f"{name:<36} {len(cols):>2} features, trained on {'measured' if real_only else 'all'} targets")
print("\nNote: the old notebook also built ramp *speed* features. Ramps never report speed in this data, so those columns would be entirely blank and are left out.")

## 4. Walk-forward validation
Each fold trains on everything from January 2023 up to the start of its validation window, then predicts the next six months. These are the same windows as the old notebook.

In [ ]:
def typical_profile(train_mask):
    t = d[train_mask & d["y_real"]]
    prof = t.groupby(["station", t["tod"] + HORIZON_MIN])["y"].median()
    idx = pd.MultiIndex.from_arrays([d["station"], d["tod"] + HORIZON_MIN])
    return pd.Series(prof.reindex(idx).to_numpy(), index=d.index, dtype="float32")

def fit_predict(cols, real_only, train_mask, eval_mask):
    tr = d[train_mask & (d["y_real"] if real_only else True)]
    if TRAIN_SAMPLE and len(tr) > TRAIN_SAMPLE:
        tr = tr.sample(TRAIN_SAMPLE, random_state=0)
    model = HistGradientBoostingRegressor(**MODEL_PARAMS).fit(tr[cols], tr["y"])
    return model, model.predict(d.loc[eval_mask, cols])

def score(pred, ev):
    err = np.abs(pred - ev["y"].to_numpy())
    ch = ev["changing"].to_numpy()
    return {"MAE": err.mean(), "MAE when changing": err[ch].mean(), "rows": len(ev), "changing rows": int(ch.sum())}

results, val_preds, last_models, last_eval = [], [], {}, None
for k, (vs, ve) in enumerate(VAL_WINDOWS, 1):
    vs, ve = pd.Timestamp(vs), pd.Timestamp(ve)
    train_mask = (d["timestamp"] >= TRAIN_START) & (d["target_time"] < vs)
    val_mask = (d["timestamp"] >= vs) & (d["target_time"] < ve)
    if train_mask.sum() == 0 or val_mask.sum() == 0:
        print(f"fold {k}: no data, skipped"); continue
    d["typical_in_30"] = typical_profile(train_mask)
    eval_mask = val_mask & d["y_real"]                      # judge on measured future speeds only
    ev = d[eval_mask]
    fold = f"{vs:%Y-%m} to {ve - pd.Timedelta(days=1):%Y-%m}"
    results.append({"fold": fold, "model": "same as now", **score(ev["speed"].to_numpy(), ev)})
    results.append({"fold": fold, "model": "typical for this time", **score(ev["typical_in_30"].fillna(ev["speed"]).to_numpy(), ev)})
    for name, p in [("same as now", ev["speed"].to_numpy()), ("typical for this time", ev["typical_in_30"].fillna(ev["speed"]).to_numpy())]:
        val_preds.append(pd.DataFrame({"fold": fold, "model": name, "hour": ev["timestamp"].dt.hour.to_numpy(),
                                       "err": np.abs(p - ev["y"].to_numpy())}))
    for name, (cols, real_only) in SETS.items():
        t0 = time.time()
        model, pred = fit_predict(cols, real_only, train_mask, eval_mask)
        results.append({"fold": fold, "model": name, **score(pred, ev)})
        if name in ("old", "new"):
            val_preds.append(pd.DataFrame({"fold": fold, "model": name, "hour": ev["timestamp"].dt.hour.to_numpy(),
                                           "err": np.abs(pred - ev["y"].to_numpy())}))
            last_models[name], last_eval = model, ev
        print(f"fold {k} ({fold}) | {name:<36} MAE {results[-1]['MAE']:.2f} | changing {results[-1]['MAE when changing']:.2f} | {time.time() - t0:.0f}s")

res = pd.DataFrame(results)
order = ["same as now", "typical for this time"] + list(SETS)
summary = (res.groupby("model")[["MAE", "MAE when changing"]].agg(["mean", "std"]).reindex(order).round(2))
summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2), sharey=True)
colors = {"same as now": "#b5b4ae", "typical for this time": "#8f8e88", "old": ORANGE, "new": BLUE}
for ax, metric in zip(axes, ["MAE", "MAE when changing"]):
    m = summary[(metric, "mean")]; s = summary[(metric, "std")]
    y = np.arange(len(order))[::-1]
    ax.barh(y, m.values, xerr=s.values, height=0.6, color=[colors.get(o, AQUA) for o in order],
            error_kw={"ecolor": INK2, "elinewidth": 1, "capsize": 3})
    for yi, v, e in zip(y, m.values, s.fillna(0).values):
        ax.annotate(f"{v:.2f}", (v + e, yi), xytext=(8, -4), textcoords="offset points", color=INK, fontsize=9)
    ax.set_xlim(0, (m + s.fillna(0)).max() * 1.15)
    ax.set_yticks(y, order); ax.set_xlabel("mph (lower is better)"); ax.set_title(metric + " (validation folds)")
    ax.grid(axis="y", visible=False)
plt.tight_layout(); plt.show()

**How to read this.** The bars are averages over the four validation folds, and the whiskers show how much they vary from fold to fold.
A change is convincing when it moves the bar by more than the whisker. Compare the **new** model with **typical for this time** in the *changing* panel:
beating it there means the model has learned something beyond the daily routine.

## 5. Where in the day the errors are
This pools the validation predictions from all four folds and shows the average error by the hour the forecast is made.

In [ ]:
vp = pd.concat(val_preds, ignore_index=True)
by_hour = vp.groupby(["model", "hour"])["err"].mean().unstack(0)
fig, ax = plt.subplots(figsize=(11, 3.8))
for name, c in [("same as now", "#b5b4ae"), ("typical for this time", "#8f8e88"), ("old", ORANGE), ("new", BLUE)]:
    ax.plot(by_hour.index, by_hour[name], color=c, marker="o", markersize=5, label=name)
ax.set_xticks(range(HOUR_START, HOUR_END + 1)); ax.set_xlabel("Hour of day (when the forecast is made)")
ax.set_ylabel("Mean absolute error (mph)"); ax.set_title("Validation error by time of day (all four folds)")
ax.legend(loc="upper left"); plt.show()

## 6. What the new model relies on
This uses permutation importance for the new model from the last validation fold (January to June 2025), on a sample of that fold's validation rows: shuffle one feature at a time and measure how much worse the predictions get. Features that carry the same information share the credit, so read it as a rough guide.

In [ ]:
sample = last_eval.sample(min(len(last_eval), 50_000), random_state=0)
imp = permutation_importance(last_models["new"], sample[NEW], sample["y"], scoring="neg_mean_absolute_error",
                             n_repeats=5, random_state=0)
order_imp = pd.Series(imp.importances_mean, index=NEW).sort_values()
fig, ax = plt.subplots(figsize=(8, 0.35 * len(NEW) + 1))
ax.barh(order_imp.index, order_imp.values, color=BLUE, height=0.6)
ax.axvline(0, color=INK2, linewidth=1)
ax.set_xlabel("Increase in MAE when shuffled (mph)"); ax.set_title("New model: feature importance (last validation fold)")
ax.grid(axis="y", visible=False); plt.tight_layout(); plt.show()

## 7. Notes

- **The holdout (July–December 2025) was never read.** Keep it that way until your final model is chosen, then evaluate on it once.
- **Measured vs. filled-in.** About 40% of 2023 targets and about a third of 2024–2025 targets are PeMS estimates. They're excluded from scoring everywhere, and from training for every set except **old**.
- **Training sample.** Each fit uses up to `TRAIN_SAMPLE` rows. Set it to `None` for a final run on everything.
- **Next ideas.** Occupancy relative to each station's critical occupancy, how far away the nearest downstream slowdown is, and forecasting the *change* in speed instead of the speed itself.